In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_predict,
    cross_val_score,
    learning_curve
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [ ]:
from pathlib import Path

# Dataset path for the repository version.
# The first path works when the notebook is run from the `notebooks/` folder;
# the second path works when it is run from the repository root.
candidate_paths = [
    Path("../dataset/dataset_additives.xlsx"),
    Path("dataset/dataset_additives.xlsx"),
    Path("dataset_additives.xlsx"),
]

dataset_path = next((path for path in candidate_paths if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Dataset not found. Expected `dataset/dataset_additives.xlsx` in the repository."
    )

print(f"Using dataset: {dataset_path}")


In [ ]:
df = pd.read_excel(
    dataset_path,
    keep_default_na=False
)

df.columns = [
    "material",
    "percentage",
    "Tg",
    "young_modulus",
    "stress",
    "strain"
]

for col in ["percentage", "Tg", "young_modulus", "stress", "strain"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

df["material"] = (
    df["material"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print(df.head())
print("\nShape dataset:", df.shape)
print("\nMaterials:", df["material"].unique())
print("\nCount by material:")
print(df["material"].value_counts(dropna=False))

In [ ]:
X_class = df[["young_modulus", "stress", "strain"]].copy()
y_class = df["material"].copy()

label_encoder = LabelEncoder()
y_class_enc = label_encoder.fit_transform(y_class)

print("Class mapping:")
for cls, code in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{cls} -> {code}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_class,
    y_class_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_class_enc
)

In [ ]:
classification_models = {
    "RandomForestClassifier_SMOTE": ImbPipeline([
        ("smote", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(
            n_estimators=150,
            max_depth=4,
            min_samples_leaf=3,
            min_samples_split=6,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "LogisticRegression_SMOTE": ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", LogisticRegression(
            C=0.5,
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "SVC_SMOTE": ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", SVC(
            C=0.7,
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ])
}

In [ ]:
from sklearn.model_selection import StratifiedKFold
from collections import Counter

class_counts = Counter(y_train)
min_class_count = min(class_counts.values())

print("Class counts in y_train:", class_counts)
print("Minimum class count:", min_class_count)

# SMOTE deve usare meno vicini della classe meno popolata
smote_k = max(1, min(3, min_class_count - 1))
print("SMOTE k_neighbors scelto:", smote_k)

classification_models = {
    "RandomForestClassifier_SMOTE": ImbPipeline([
        ("smote", SMOTE(random_state=42, k_neighbors=smote_k)),
        ("model", RandomForestClassifier(
            n_estimators=150,
            max_depth=4,
            min_samples_leaf=3,
            min_samples_split=6,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "LogisticRegression_SMOTE": ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42, k_neighbors=smote_k)),
        ("model", LogisticRegression(
            C=0.5,
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "SVC_SMOTE": ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42, k_neighbors=smote_k)),
        ("model", SVC(
            C=0.7,
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ])
}

In [ ]:
classification_results = []
trained_classifiers = {}

best_clf_name = None
best_clf_model = None
best_clf_score = -np.inf
best_y_pred = None

print("=== CLASSIFICATION MODELS ===")

# Meglio di KFold per classificazione sbilanciata
n_splits_clf = max(2, min(5, min_class_count))
cv = StratifiedKFold(n_splits=n_splits_clf, shuffle=True, random_state=42)

print("Numero fold classificazione:", n_splits_clf)

for name, base_model in classification_models.items():
    model = clone(base_model)

    try:
        cv_scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="accuracy",
            error_score=np.nan
        )
        cv_acc = np.nanmean(cv_scores)

        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        train_acc = accuracy_score(y_train, train_pred)
        test_acc = accuracy_score(y_test, test_pred)

        gap = train_acc - cv_acc if not np.isnan(cv_acc) else np.nan
        penalized_score = cv_acc - abs(gap) * 0.5 if not np.isnan(gap) else -np.inf

        print(
            f"{name:30} -> CV Acc = {cv_acc:.4f} | "
            f"Train Acc = {train_acc:.4f} | Test Acc = {test_acc:.4f} | "
            f"Gap = {gap:.4f} | Penalized = {penalized_score:.4f}"
        )

        classification_results.append({
            "classifier": name,
            "cv_accuracy": cv_acc,
            "train_accuracy": train_acc,
            "test_accuracy": test_acc,
            "gap": gap,
            "penalized_score": penalized_score
        })

        trained_classifiers[name] = model

        if penalized_score > best_clf_score:
            best_clf_score = penalized_score
            best_clf_name = name
            best_clf_model = model
            best_y_pred = test_pred

    except Exception as e:
        print(f"{name:30} -> ERRORE: {e}")

classification_results_df = pd.DataFrame(classification_results)

if not classification_results_df.empty:
    classification_results_df = classification_results_df.sort_values(
        "penalized_score", ascending=False
    )

print("\nBEST CLASSIFIER:", best_clf_name)
print(f"Best penalized score: {best_clf_score:.4f}")

classification_results_df

In [ ]:
print("=== CONFUSION MATRIX ===")

if best_y_pred is None:
    print("Nessun classificatore valido disponibile.")
else:
    y_test_arr = np.asarray(y_test).ravel().astype(int)
    best_y_pred_arr = np.asarray(best_y_pred).ravel().astype(int)

    labels = list(range(len(label_encoder.classes_)))
    class_names = [str(x) for x in label_encoder.classes_]

    cm = confusion_matrix(
        y_test_arr,
        best_y_pred_arr,
        labels=labels
    )

    print("\nMatrice di confusione (valori):")
    print(cm)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix - {best_clf_name}")
    plt.show()

In [ ]:
df_NA = df[df["material"] == "NA"].copy()
df_MW = df[df["material"] == "MW"].copy()
df_OW = df[df["material"] == "OW"].copy()

print("NA shape:", df_NA.shape)
print("MW shape:", df_MW.shape)
print("OW shape:", df_OW.shape)

In [ ]:
regression_models = {
    "Linear": Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ]),

    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=3.0))
    ]),

    "Polynomial": Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("ridge", Ridge(alpha=8.0))
    ]),

    "RandomForest": RandomForestRegressor(
        n_estimators=150,
        max_depth=4,
        min_samples_leaf=4,
        min_samples_split=8,
        max_features="sqrt",
        random_state=42
    ),

    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=80,
        learning_rate=0.03,
        max_depth=2,
        min_samples_leaf=3,
        subsample=0.85,
        random_state=42
    )
}

In [ ]:
datasets_by_material = {
    "MW": df_MW,
    "OW": df_OW
}

trained_regressors = {}
regression_metrics = {}
best_regressor_name = {}

for material_name, df_sub in datasets_by_material.items():

    print(f"\n==============================")
    print(f"REGRESSION FOR MATERIAL: {material_name}")
    print(f"==============================")

    trained_regressors[material_name] = {}
    regression_metrics[material_name] = []

    X_sub = df_sub[["young_modulus", "stress", "strain"]].copy()
    y_sub = df_sub["percentage"].copy()

    best_rmse = np.inf
    best_name = None

    for model_name, base_model in regression_models.items():

        cv_model = clone(base_model)

        kf = KFold(
            n_splits=min(5, len(X_sub)),
            shuffle=True,
            random_state=42
        )

        preds = cross_val_predict(cv_model, X_sub, y_sub, cv=kf)

        rmse = np.sqrt(mean_squared_error(y_sub, preds))
        mae = mean_absolute_error(y_sub, preds)
        r2 = r2_score(y_sub, preds)

        # modello finale addestrato su tutto il dataset del materiale
        final_model = clone(base_model)
        final_model.fit(X_sub, y_sub)
        trained_regressors[material_name][model_name] = final_model

        train_pred = final_model.predict(X_sub)
        train_rmse = np.sqrt(mean_squared_error(y_sub, train_pred))
        gap = rmse - train_rmse
        gap_rel = gap / rmse if rmse > 0 else np.nan

        regression_metrics[material_name].append({
            "model": model_name,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "train_rmse": train_rmse,
            "gap": gap,
            "gap_rel": gap_rel
        })

        print(
            f"{model_name:20} -> RMSE={rmse:.2f} | MAE={mae:.2f} | "
            f"R2={r2:.3f} | Train_RMSE={train_rmse:.2f} | "
            f"Gap={gap:.2f} | Gap_rel={100*gap_rel:.1f}%"
        )

        # selezione primaria sul CV RMSE
        if rmse < best_rmse:
            best_rmse = rmse
            best_name = model_name

    # tie-break: se più modelli sono vicini, scegli quello col gap minore
    metrics_df = pd.DataFrame(regression_metrics[material_name]).sort_values("rmse")
    best_rmse_value = metrics_df.iloc[0]["rmse"]
    tolerance = 0.10 * best_rmse_value  # 10% di tolleranza

    candidate_df = metrics_df[metrics_df["rmse"] <= best_rmse_value + tolerance]
    best_name = candidate_df.sort_values(["gap", "mae"]).iloc[0]["model"]

    best_regressor_name[material_name] = best_name

    selected_row = candidate_df.sort_values(["gap", "mae"]).iloc[0]
    print(
        f"\nBEST REGRESSOR for {material_name}: {best_name} "
        f"(RMSE={selected_row['rmse']:.2f}, Gap={selected_row['gap']:.2f})"
    )

In [ ]:
all_regression_rows = []

for material_name, metrics_list in regression_metrics.items():
    for row in metrics_list:
        row_copy = row.copy()
        row_copy["material"] = material_name
        all_regression_rows.append(row_copy)

regression_summary_df = pd.DataFrame(all_regression_rows)
regression_summary_df = regression_summary_df.sort_values(
    ["material", "rmse", "gap"],
    ascending=[True, True, True]
)

regression_summary_df

In [ ]:
def plot_learning_curve_for_regressor(model, X, y, title):
    cv = KFold(
        n_splits=min(5, len(X)),
        shuffle=True,
        random_state=42
    )

    train_sizes, train_scores, val_scores = learning_curve(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        train_sizes=np.linspace(0.4, 1.0, 5),
        n_jobs=None
    )

    train_rmse = -train_scores
    val_rmse = -val_scores

    train_mean = train_rmse.mean(axis=1)
    train_std = train_rmse.std(axis=1)

    val_mean = val_rmse.mean(axis=1)
    val_std = val_rmse.std(axis=1)

    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_mean, marker="o", label="Train RMSE")
    plt.plot(train_sizes, val_mean, marker="s", label="Validation RMSE")

    plt.fill_between(
        train_sizes,
        train_mean - train_std,
        train_mean + train_std,
        alpha=0.2
    )
    plt.fill_between(
        train_sizes,
        val_mean - val_std,
        val_mean + val_std,
        alpha=0.2
    )

    plt.xlabel("Training set size")
    plt.ylabel("RMSE")
    plt.title(title)
    plt.grid(True, linestyle=":")
    plt.legend()
    plt.show()

In [ ]:
from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.base import clone
from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np
import pandas as pd

# scorer RMSE positivo (cross_validate massimizza, quindi usiamo negativo e poi invertiamo)
rmse_scorer = make_scorer(
    lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=False
)

overfitting_report = []

for material_name, df_sub in datasets_by_material.items():

    best_name = best_regressor_name[material_name]
    best_model = clone(regression_models[best_name])

    X_sub = df_sub[["young_modulus", "stress", "strain"]].copy()
    y_sub = df_sub["percentage"].copy()

    n_samples = len(X_sub)

    # numero fold sicuro per dataset piccoli
    n_splits = min(5, max(2, n_samples))

    cv = RepeatedKFold(
        n_splits=n_splits,
        n_repeats=10,
        random_state=42
    )

    scores = cross_validate(
        best_model,
        X_sub,
        y_sub,
        cv=cv,
        scoring=rmse_scorer,
        return_train_score=True,
        n_jobs=-1
    )

    # invertiamo il segno
    train_rmse = -scores["train_score"]
    val_rmse   = -scores["test_score"]

    train_mean = train_rmse.mean()
    train_std  = train_rmse.std()

    val_mean = val_rmse.mean()
    val_std  = val_rmse.std()

    gap_abs = val_mean - train_mean
    gap_rel = gap_abs / val_mean if val_mean > 0 else np.nan

    target_range = y_sub.max() - y_sub.min()
    gap_norm_range = gap_abs / target_range if target_range > 0 else np.nan

    # rischio overfitting
    if gap_rel < 0.15:
        risk = "low"
    elif gap_rel < 0.30:
        risk = "moderate"
    else:
        risk = "high"

    # stabilità modello
    stability = val_std / val_mean if val_mean > 0 else np.nan

    if stability < 0.10:
        stability_label = "stable"
    elif stability < 0.25:
        stability_label = "moderate"
    else:
        stability_label = "unstable"

    overfitting_report.append({
        "material": material_name,
        "best_model": best_name,

        "train_rmse_mean": train_mean,
        "train_rmse_std": train_std,

        "cv_rmse_mean": val_mean,
        "cv_rmse_std": val_std,

        "gap_abs": gap_abs,
        "gap_rel": gap_rel,
        "gap_norm_range": gap_norm_range,

        "overfitting_risk": risk,
        "model_stability": stability_label
    })

overfitting_report_df = pd.DataFrame(overfitting_report)

# ordinamento opzionale
overfitting_report_df = overfitting_report_df.sort_values(
    by=["overfitting_risk", "cv_rmse_mean"]
).reset_index(drop=True)

overfitting_report_df

In [ ]:
for material_name, df_sub in datasets_by_material.items():
    best_name = best_regressor_name[material_name]
    best_model = clone(regression_models[best_name])

    X_sub = df_sub[["young_modulus", "stress", "strain"]].copy()
    y_sub = df_sub["percentage"].copy()

    print(f"\nLearning curve - {material_name} - {best_name}")
    plot_learning_curve_for_regressor(
        best_model,
        X_sub,
        y_sub,
        title=f"Learning Curve - {material_name} - {best_name}"
    )

In [ ]:
print("=== FINAL OVERFITTING CHECK ===")

for _, row in overfitting_report_df.iterrows():
    print(
        f"{row['material']} | {row['best_model']} | "
        f"train_RMSE={row['train_rmse_mean']:.3f} ± {row['train_rmse_std']:.3f} | "
        f"CV_RMSE={row['cv_rmse_mean']:.3f} ± {row['cv_rmse_std']:.3f} | "
        f"gap_abs={row['gap_abs']:.3f} | "
        f"gap_rel={100*row['gap_rel']:.1f}% | "
        f"gap/range={100*row['gap_norm_range']:.1f}% | "
        f"risk={row['overfitting_risk']} | "
        f"stability={row['model_stability']}"
    )

high_count = (overfitting_report_df["overfitting_risk"] == "high").sum()
moderate_count = (overfitting_report_df["overfitting_risk"] == "moderate").sum()

if high_count > 0:
    print("\nFINAL RESULT: at least one regressor shows a high overfitting risk.")
elif moderate_count > 0:
    print("\nFINAL RESULT: no severe overfitting signal, but at least one regressor shows moderate risk.")
else:
    print("\nFINAL RESULT: no strong overfitting signal detected.")

In [ ]:
def predict_material_and_percentage(young, stress, strain):
    X_input = pd.DataFrame(
        [[young, stress, strain]],
        columns=["young_modulus", "stress", "strain"]
    )

    print("\n=== INPUT RECEIVED ===")
    print(X_input)

    pred_class_enc = best_clf_model.predict(X_input)[0]
    pred_material = label_encoder.inverse_transform([pred_class_enc])[0]

    if hasattr(best_clf_model, "predict_proba"):
        probs = best_clf_model.predict_proba(X_input)[0]
        prob_map = {
            label_encoder.inverse_transform([i])[0]: p
            for i, p in enumerate(probs)
        }
    else:
        prob_map = None

    print("\n=== CLASSIFICATION RESULT ===")
    print(f"Best classifier: {best_clf_name}")
    print(f"Predicted material: {pred_material}")

    if prob_map is not None:
        print("Class probabilities:")
        for mat, p in prob_map.items():
            print(f"  {mat}: {p:.3f}")

    if pred_material == "NA":
        print("\n=== REGRESSION RESULT ===")
        print("Material NA detected -> no additive -> percentage fixed at 0.00%")

        return pd.DataFrame([{
            "predicted_material": "NA",
            "model": "Constant",
            "predicted_percentage": 0.0,
            "rmse": np.nan,
            "mae": np.nan,
            "r2": np.nan,
            "train_rmse": np.nan,
            "gap": np.nan,
            "gap_rel": np.nan
        }])

    print("\n=== REGRESSION RESULTS ===")
    print(f"Using regressors trained only on: {pred_material}")

    all_predictions = []

    material_df = datasets_by_material[pred_material]
    min_val = material_df["percentage"].min()
    max_val = material_df["percentage"].max()

    for model_name in regression_models.keys():
        metric = next(
            m for m in regression_metrics[pred_material]
            if m["model"] == model_name
        )

        model = trained_regressors[pred_material][model_name]
        pred_pct = model.predict(X_input)[0]
        pred_pct = np.clip(pred_pct, min_val, max_val)

        r2_text = f"{metric['r2']:.3f}" if pd.notnull(metric["r2"]) else "NaN"
        gap_rel_text = f"{100*metric['gap_rel']:.1f}%" if pd.notnull(metric["gap_rel"]) else "NaN"

        print(
            f"{model_name:20} -> Predicted percentage = {pred_pct:.2f}% | "
            f"RMSE={metric['rmse']:.2f} | MAE={metric['mae']:.2f} | "
            f"R2={r2_text} | Train_RMSE={metric['train_rmse']:.2f} | "
            f"Gap={metric['gap']:.2f} | Gap_rel={gap_rel_text}"
        )

        all_predictions.append({
            "predicted_material": pred_material,
            "model": model_name,
            "predicted_percentage": pred_pct,
            "rmse": metric["rmse"],
            "mae": metric["mae"],
            "r2": metric["r2"],
            "train_rmse": metric["train_rmse"],
            "gap": metric["gap"],
            "gap_rel": metric["gap_rel"]
        })

    best_reg_name = best_regressor_name[pred_material]
    best_row = next(row for row in all_predictions if row["model"] == best_reg_name)

    print("\n=== BEST REGRESSOR RESULT ===")
    print(f"Best regressor for {pred_material}: {best_reg_name}")
    print(f"Best predicted percentage: {best_row['predicted_percentage']:.2f}%")

    overfit_row = overfitting_report_df[
        overfitting_report_df["material"] == pred_material
    ].iloc[0]

    print(
        f"Overfitting risk for best regressor: {overfit_row['overfitting_risk']} "
        f"(CV_RMSE={overfit_row['cv_rmse_mean']:.3f} ± {overfit_row['cv_rmse_std']:.3f}, "
        f"gap_rel={100*overfit_row['gap_rel']:.1f}%)"
    )

    return pd.DataFrame(all_predictions).sort_values(["rmse", "gap"])

In [ ]:
results_input = predict_material_and_percentage(
    young=3.89,
    stress=47.46,
    strain=1.80
)

results_input

In [ ]:
results_input = predict_material_and_percentage(
    young=1.5,
    stress=35,
    strain=3
)

results_input

In [ ]:
results_input = predict_material_and_percentage(
    young=1.8,
    stress=35,
    strain=3.5
)

results_input

In [ ]:
results_input = predict_material_and_percentage(
    young=3.5,
    stress=70,
    strain=3.5
)

results_input

In [ ]:
results_input = predict_material_and_percentage(
    young=3.41,
    stress=65.5,
    strain=2.9
)

results_input

In [ ]:
results_input = predict_material_and_percentage(
    young=3.93,
    stress=54.5,
    strain=2.64
)

results_input

In [ ]:
results_input = predict_material_and_percentage(
    young=2.2,
    stress=37,
    strain=3.1
)

results_input